# Model-to-Data: Comparing Databases of Ground Motion Observations an Physics-Based Simulations

In this notebook we will present the workflow used in the example application of the Zaccarelli et al. (2025) paper presenting eGSIM. This is a more advanced example of how to use eGSIM as a basis for comparing databases of ground motions from physics-based observations against databases of simulations.

The objectives are as follows:

1) How to run a "rankings" analysis to retrieve fit-to-data scores for a selection of GMMs
   
2) Retreive the normalized random effects residuals (between-event $\delta B_E$ and within-event $\delta W$) for the same data set and ground motion models and plot these with respect to different predictor variables

3) Use the eGSIM API and pandas to make estimates of the site-to-site residuals $\delta S2S_S$


The data set of observed ground motions is adapted from a sub-set the Engineering Strong Motion (ESM) Flatfile (Lanzano et al. 2018) available from https://esm-db.eu/#/products/flat_file. A separate flatfile has been compiled for the selected events using data available from the Engineering Strong Motion Database (Luzi et al., 2020).

The dataset and flatfile of physics-based ground motion simuluations is the BB-SPEEDset (version 2.3) originally published by Paolucci et al. (2021). 

```
Lanzano, G., Sgobba, S., Luzi, L., Puglia, R., Pacor, F., Felicetta, C., D’Amico, M., Cotton, F., & Bindi, D. (2019). The pan-European Engineering Strong Motion (ESM) flatfile: Compilation criteria and data statistics. Bulletin of Earthquake Engineering, 17(2), 561–582. https://doi.org/10.1007/s10518-018-0480-z

Luzi L., Lanzano G., Felicetta C., D’Amico M. C., Russo E., Sgobba S., Pacor, F., & ORFEUS Working Group 5 (2020). Engineering Strong Motion Database (ESM) (Version 2.0). Istituto Nazionale di Geofisica e Vulcanologia (INGV). https://doi.org/10.13127/ESM.2

Paolucci, R., Smerzini, C., and Vanini, M. (2021). BB-SPEEDset: A validated dataset of broadband near-source earthquake ground motions from 3d physics-based numerical simulations. Bulletin of the Seismological Society of America, 111(5), 2527–2545. https://doi.org/10.1785/0120210089
```

Tools for use here:

In [ ]:
%matplotlib inline
import requests  # To make the eGSIM API request
from typing import List, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LogNorm
import seaborn as sns  # For some "prettier" plotting functionalities

Function to get the residuals using the eGSIM API:

In [ ]:
def get_residuals_from_egsim(
    flatfile_path: str,
    gmms: List,
    imts: List,
    data_format: str = "hdf",
    query_string: str = "",
    normalize: bool = True,
    likelihood: bool = False,
    ranking: bool = False
) -> Dict:
    """Retreive the residuals for the flatfile and the selected
    set of ground motion models and intensity measure types

    Args:
        flatfile_path: Local path to the selected flatfile
        gmms: List of ground motion models (OpenQuake class names)
        imts: List of intensity measure types (e.g. PGA, PGV, SA(0.1) etc.)
        plot_type: Column to return for x-values (e.g. mag, rrup, etc)
        query_string: Selection query to apply to the data

    Returns:
        json_response: Response of the POST request in json form (if successful)
                       or the response class (if unsuccessful)
    """
    assert data_format in ("hdf", "csv"), "Data format must be either 'hdf' or 'csv'"
    # Set the eGSIM URL to get the ground motion residuals
    egsim_url_residuals = "https://egsim.gfz-potsdam.de/api/query/residuals"
    # Retreive residuals for a given plot type
    parameters = {
            "model": gmms,
            "imt": imts,
            "format": data_format      
    }
    if query_string:
        parameters["data-query"] = query_string
    if not normalize:
        parameters["normalize"] = False
    if likelihood:
        parameters["likelihood"] = True
    if ranking:
        parameters["ranking"] = True

    with open(flatfile_path, "rb") as flatfile:
        files = {"flatfile": flatfile}
        try:
            # POST request for eGSIM
            response = requests.post(
                egsim_url_residuals,
                files=files,
                data=parameters
            )
            print(response.status_code)
            response.raise_for_status()
        except requests.exceptions.HTTPError as exc:
            code = exc.response.status_code
            print(response.text)
            msg = response.json()['message']
            print(f"HTTPError (code={code}): {msg}")
        except:
            print("Response failed - see status message")
            raise  
    
    if parameters['format'] == 'hdf':
        # `pd.read_hdf` works for HDF files on disk. Workaround:
        with pd.HDFStore(
                "data.h5",  # apparently unused for in-memory data
                mode="r",
                driver="H5FD_CORE",  # create in-memory file
                driver_core_backing_store=0,  # for safety, just in case
                driver_core_image=response.content) as store:
            dframe = store[list(store.keys())[0]]
    else:
        # use `pd.read_csv` with a BytesIO (file-like object) as input: 
        dframe = pd.read_csv(io.BytesIO(response.content), header=[0, 1, 2], index_col=0)        

    return dframe

# Setup the data sets and analysis configuration

In this analysis we will consider four GMMs (the same four used in the Model-to-Data demonstration notebook):

1) Bindi et al. (2014) - Using the Joyner-Boore Distance Metric (`BindiEtAl2014Rjb`)
2) Cauzzi et al. (2015) - Calibrated on Japanese/global data (`CauzziEtAl2014`)
3) Chiou & Youngs (2014) - NGA West GMM calibrated on Western US and Global Data (`ChiouYoungs2014`)
4) Kotha et al. (2020) (ESHM20) - Version of the Kotha et al. (2020) GMM originally fit to ESM data, with adjustments made for ESHM20 by Weatherill et al. (2020) (`KothaEtAl2020ESHM20`)

For the intensity measures we will consider PGA and PGV, then for spectral acceleration we define a set of periods between 0.05 s (the shortest period for data in the BB-SPEEDset) and 3.0 s (the longest period considerdd by the Bindi et al., (2014) GMM). The periods are not equally spaced but rather distributed to better sample the acceleration spectrum.

In this analysis we will consider only the normalized residuals.

In [ ]:
gmms = {
    "BindiEtAl2014Rjb": "Bindi et al. (2014)",
    "CauzziEtAl2014": "Cauzzi et al. (2015)",
    "ChiouYoungs2014": "Chiou & Youngs (2014)",
    "KothaEtAl2020ESHM20": "ESHM20"
}

periods = [0.05, 0.075, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.5, 2.0, 2.5, 3.0]
imts = ['PGV', 'PGA'] + [f"SA({per:.3f})" for per in periods]

Most of the physics-based simulations are limited to larger magnitudes and distances shorter than 100 km. Here we limit both datasets to events with magnitude greater than 5.0, rupture distances less than 120 km and hypocentral depths shallower than 35 km

In [ ]:
# This query may take a minute or so
residuals_observations = get_residuals_from_egsim(
    "../data/flatfiles/esm_sample_for_demo.csv",
    gmms=list(gmms),
    imts=imts,
    query_string="(rrup <= 120.0) & (mag >= 5.0) & (evt_depth <= 35.0)",
    normalize=True,
)
residuals_observations

In [ ]:
# This query might take a minute or so
residuals_simulations = get_residuals_from_egsim(
    "../data/flatfiles/bbspeedset_sample_for_demo.csv", 
    gmms=list(gmms),
    imts=imts,
    query_string="(rrup <= 120.0) & (mag >= 5.0) & (evt_depth <= 35.0)",
    normalize=True
)
residuals_simulations

In [ ]:
# Show the distribution of the datasets
from matplotlib.colors import Normalize, LogNorm
fig, axs = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
# Observations
x_column = "input distance_measure rrup"
y_column = "input rupture_parameter mag"
c_column = "input site_parameter vs30"
# Create a scatter plot of the observations (magnitude, distance and Vs30)
cax0 = axs[0].scatter(
    residuals_observations[x_column],
    residuals_observations[y_column],
    c=residuals_observations[c_column],
    cmap="magma",
    norm=Normalize(200.,1200.),
    edgecolor="k",
    linewidth=0.5, s=50
)
axs[0].set_title("Observations (ESM)", fontsize=20)
axs[0].set_ylabel(r"$M_W$", fontsize=16)

# Create a scstter plot of the simulations
cax1 = axs[1].scatter(
    residuals_simulations[x_column],
    residuals_simulations[y_column],
    c=residuals_simulations[c_column],
    cmap="magma",
    norm=Normalize(200.,1200.),
    edgecolor="k",
    linewidth=0.5, s=60
)
axs[1].set_title("Simulations (BB-SPEEDset)", fontsize=20)

# Generl axes control
for ax in axs:
    ax.grid(which="both", zorder=1)
    ax.set_xscale("log")
    ax.set_xlim(0.8, 200.0)
    ax.set_ylim(5.0, 8.0)
    ax.set_xlabel(r"$R_{RUP}$ (km)", fontsize=16)
    ax.set_xticks([1, 10, 100], ["1", "10", "100"])
    ax.tick_params(labelsize=12)

fig.tight_layout()
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.91, 0.11, 0.03, 0.8])
cbar1 = fig.colorbar(cax1, cax=cbar_ax, label=r"$V_{S30}$ (m/s)", pad=0.02, )

# To create the plot for Figure 4
# plt.savefig(
#     "./Figure_4_comparison_ESM_SPEED_datasets.jpg",
#     format="jpg",
#     dpi=300,
#     bbox_inches="tight"
# )

### Compare the distributions of between-event residuals ($\delta B_e$)

In [ ]:
def dataframe_for_seaborn(
        residuals: pd.DataFrame,
        gmms: Dict,
        selected_imts: List,
        residual_type: str,
        input_parameter: str,
        source_type: str,
        y_column_name: str = "",
        x_column_name: str = ""
) -> pd.DataFrame:
    """Converts a dataframe output from an eGSIM residuals query into
    a format useful for seaborn.

    Args:
        residuals: The dataframe output from the eGSIM query
        gmms: A dictionary of GMMs with each entry indicate by the OpenQuake
              class name and the value a tuple of a "pretty" name and plotting color
        selected_imts: The chosen IMTs for visualisation
        residual_type: The choice of "inter_event_residual", "intra_event_residual" or
                       "total_residual"
        input_paramter: The x-variable for the regression plots
        source_type: Label for all the observations
        y_column_name: Optional name to call the y-variable in the dataframe
        x_column_name: Optional name to call the x-variable in the dataframe

    Returns:
        Dataframe for seaborn visualization.
    """
    full_dataframe = []
    xcolumn = f"input {input_parameter}"
    for i, imt in enumerate(selected_imts):
        for j, (gmm, gmm_label) in enumerate(gmms.items()):
            ycolumn = f"{imt} {residual_type} {gmm}"
            xyvals = residuals[[xcolumn, ycolumn]].drop_duplicates(
                [xcolumn, ycolumn],
                inplace=False,
                ignore_index=True
            )
            column_mapping = {}
            if x_column_name:
                column_mapping[xcolumn] = x_column_name
            if y_column_name:
                column_mapping[ycolumn] = y_column_name
            if column_mapping:
                xyvals.rename(columns=column_mapping,
                              inplace=True,
                              copy=False)
            xyvals["IMT"] = [imt] * xyvals.shape[0]
            xyvals["GMM"] = [gmm_label] * xyvals.shape[0]
            xyvals["Type"] = [source_type] * xyvals.shape[0]
            full_dataframe.append(xyvals)
    return pd.concat(full_dataframe, axis=0, ignore_index=True)

In [ ]:
# Setup the seaborne dataframe for the residuals of the observations 
observation_dataframe = dataframe_for_seaborn(
    residuals=residuals_observations,
    gmms=gmms,
    selected_imts=["PGV", "PGA", "SA(0.2)", "SA(2.0)"],
    residual_type="inter_event_residual",
    input_parameter="rupture_parameter mag",
    source_type="Observations",
    y_column_name="dBe",
    x_column_name="Mw",
)
# Setup the seaborne dataframe for the residuals of the simulations
simulation_dataframe = dataframe_for_seaborn(
    residuals=residuals_simulations,
    gmms=gmms,
    selected_imts=["PGV", "PGA", "SA(0.2)", "SA(2.0)"],
    residual_type="inter_event_residual",
    input_parameter="rupture_parameter mag",
    source_type="Simulations",
    y_column_name="dBe",
    x_column_name="Mw",
)

# Join the two dataframes 
full_dataframe = pd.concat(
    [observation_dataframe, simulation_dataframe],
    axis=0,
    ignore_index=True
)
# Plot the distributions separated by type
with sns.plotting_context("paper", font_scale=2.0):
    with sns.axes_style("darkgrid", ):
        fgrid = sns.lmplot(full_dataframe,
                           x="Mw",
                           y="dBe",
                           hue="Type",
                           col="GMM",
                           row="IMT",
                           fit_reg=True,
                           markers=["o", "s"],
                           palette=["tab:blue", "tab:orange"],
                           scatter_kws={"s": 70},
                           facet_kws={"sharex": True, "sharey":True}, )
        fgrid.set(xlim=(5.0, 8.0), ylim=(-3.5, 3.5))
        fgrid.set_axis_labels(r"$M_W$", r"$\delta B_e$")
        for ax in fgrid.axes.flatten():
            ax.title.set_text(ax.title.get_text().replace("|", "\n"))
            ax.set_xticks([5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0],
                          ["5.0", "5.5", "6.0", "6.5", "7.0", "7.5", "8.0"])
            ax.set_yticks([-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0],
                          ["-3", "-2", "-1", "0", "1", "2", "3"])
        fgrid.tight_layout()

# To create the plot for Figure 5
# plt.savefig(
#     "./Figure_5_dBe_comparison_ESM_SPEED_datasets.jpg",
#     format="jpg",
#     dpi=300,
#     bbox_inches="tight"
# )

### Compare Distributions of Within-event residuals ($\delta W$) with respect to distance

In [ ]:
# Setup the seaborne dataframe for the residuals of the observations 
observation_dataframe = dataframe_for_seaborn(
    residuals=residuals_observations,
    gmms=gmms,
    selected_imts=["PGV", "PGA", "SA(0.2)", "SA(2.0)"],
    residual_type="intra_event_residual",
    input_parameter="distance_measure rrup",
    source_type="Observations",
    y_column_name="dW",
    x_column_name="rrup",
)
# Setup the seaborne dataframe for the residuals of the simulations
simulation_dataframe = dataframe_for_seaborn(
    residuals=residuals_simulations,
    gmms=gmms,
    selected_imts=["PGV", "PGA", "SA(0.2)", "SA(2.0)"],
    residual_type="intra_event_residual",
    input_parameter="distance_measure rrup",
    source_type="Simulations",
    y_column_name="dW",
    x_column_name="rrup",
)

# Join the two dataframes 
full_dataframe = pd.concat(
    [observation_dataframe, simulation_dataframe],
    axis=0,
    ignore_index=True
)
# Plot the distributions separated by type
with sns.plotting_context("paper", font_scale=2.0):
    with sns.axes_style("darkgrid", ):
        fgrid = sns.lmplot(full_dataframe,
                           x="rrup",
                           y="dW",
                           hue="Type",
                           col="GMM",
                           row="IMT",
                           fit_reg=True,
                           markers=["o", "s"],
                           palette=["tab:blue", "tab:orange"],
                           scatter_kws={"alpha": 0.8, "zorder": 2},
                           line_kws={"linewidth": 3.0, "zorder": 4, "antialiased": True},
                          )
        fgrid.set(xlim=(0.0, 120.0), ylim=(-3.5, 3.5), xscale="linear")
        fgrid.set_axis_labels(r"$R_{RUP}$ (km)", r"$\delta W$")
        for ax in fgrid.axes.flatten():
            ax.title.set_text(ax.title.get_text().replace("|", "\n"))
            ax.set_yticks([-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0],
                          ["-3", "-2", "-1", "0", "1", "2", "3"])
        fgrid.tight_layout()

# To create the plot for Figure 6
# plt.savefig(
#     "./Figure_6_dW_comparison_ESM_SPEED_datasets.jpg",
#     format="jpg",
#     dpi=300,
#     bbox_inches="tight"
# )

### What about the distribution of within-event residuals with respect to magnitude?

In [ ]:
# Setup the seaborne dataframe for the residuals of the observations 
observation_dataframe = dataframe_for_seaborn(
    residuals=residuals_observations,
    gmms=gmms,
    selected_imts=["PGV", "PGA", "SA(0.2)", "SA(2.0)"],
    residual_type="intra_event_residual",
    input_parameter="rupture_parameter mag",
    source_type="Observations",
    y_column_name="dW",
    x_column_name="Mw",
)
# Setup the seaborne dataframe for the residuals of the simulations
simulation_dataframe = dataframe_for_seaborn(
    residuals=residuals_simulations,
    gmms=gmms,
    selected_imts=["PGV", "PGA", "SA(0.2)", "SA(2.0)"],
    residual_type="intra_event_residual",
    input_parameter="rupture_parameter mag",
    source_type="Simulations",
    y_column_name="dW",
    x_column_name="Mw",
)

# Join the two dataframes 
full_dataframe = pd.concat(
    [observation_dataframe, simulation_dataframe],
    axis=0,
    ignore_index=True
)
# Plot the distributions separated by type
with sns.plotting_context("paper", font_scale=2.0):
    with sns.axes_style("darkgrid", ):
        fgrid = sns.lmplot(full_dataframe,
                           x="Mw",
                           y="dW",
                           hue="Type",
                           col="GMM",
                           row="IMT",
                           fit_reg=True,
                           markers=["o", "s"],
                           palette=["tab:blue", "tab:orange"],
                           scatter_kws={"alpha": 0.8, "zorder": 2},
                           line_kws={"linewidth": 3.0, "zorder": 4, "antialiased": True},
                          )
        fgrid.set(xlim=(5.0, 8.0), ylim=(-3.5, 3.5), xscale="linear")
        fgrid.set_axis_labels(r"$M_W$ (km)", r"$\delta W$")
        for ax in fgrid.axes.flatten():
            ax.title.set_text(ax.title.get_text().replace("|", "\n"))
            ax.set_yticks([-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0],
                          ["-3", "-2", "-1", "0", "1", "2", "3"])
        fgrid.tight_layout()

# Can we look more closely at the random-effects residuals for a single event?

The trends in the residuals show clear differences in the two data sets in terms of the distribution of between-event residuals $\delta B_e$, with a the simulations showing a clear trend of increasing $\delta B_e$ with magnitude. To get a deeper understanding as to why this might be the case it can be insightful to look at individual events. The BB-SPEEDset contains simulations from earthquakes that are a mixture of simulations of previous earthquakes and simulations from hypothetical earthquake scenarios. Within the BB-SPEEDset we can identify five earthquakes for which we can find observations of shaking within the ESM database:

1) Irpinia (1980), Italy $M_W$ 6.9
2) L'Aquila (2009), Italy $M_W$ 6.1
3) Emilia-Romagna (2012) [1st shock], Italy, $M_W$ 6.0
4) Central Italy (2016) [2nd shock], Italy, $M_W$ 6.6
5) Sea-of-Marmara (2019), Turkiye, $M_W$ 5.9

We consider the first four events (#5 is discarded owing to a minimal overlap between observation locations and simulation receivers) and have taken processed the records from ESM. For convenience, the finite-fault rupture planes adopted by the BB-SPEEDset is used to determine the finite fault metrics for the observed stations.

Both the simulation and observation data sets are limited to stations/recievers within 50 km of the rupture plane.

This comparison highlights the sort of analysis that can be undertaken for just a single event, which shows how eGSIM could be used in a rapid post-event context.

In [ ]:
# This query may take a minute or so
residuals_observations = get_residuals_from_egsim(
    "../data/flatfiles/observations_esm_selected_earthquakes.csv", 
    gmms=list(gmms),
    imts=imts,
    query_string="(rrup <= 50.0)",
    normalize=True,
)
residuals_observations

In [ ]:
residuals_simulations = get_residuals_from_egsim(
    "../data/flatfiles/simulations_bbspeedset_selected_earthquakes.csv",
    gmms=list(gmms),
    imts=imts,
    query_string="(rrup <= 50.0)",
    normalize=True
)
residuals_simulations

In [ ]:
# Show the distribution of the datasets
fig, axs = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
# Observations
x_column = "input distance_measure rrup"
y_column = "input rupture_parameter mag"
c_column = "input site_parameter vs30"
# Create a scatter plot of the observations (magnitude, distance and Vs30)
cax0 = axs[0].scatter(
    residuals_observations[x_column],
    residuals_observations[y_column],
    c=residuals_observations[c_column],
    cmap="magma",
    norm=Normalize(200.,1200.),
    edgecolor="k",
    linewidth=0.5, s=50
)
axs[0].set_title("Observations (ESM)", fontsize=20)
axs[0].set_ylabel(r"$M_W$", fontsize=16)

# Create a scstter plot of the simulations
cax1 = axs[1].scatter(
    residuals_simulations[x_column],
    residuals_simulations[y_column],
    c=residuals_simulations[c_column],
    cmap="magma",
    norm=Normalize(200.,1200.),
    edgecolor="k",
    linewidth=0.5, s=60
)
axs[1].set_title("Simulations (BB-SPEEDset)", fontsize=20)

# Generl axes control
for ax in axs:
    ax.grid(which="both", zorder=1)
    ax.set_xscale("log")
    ax.set_xlim(0.8, 200.0)
    ax.set_ylim(5.75, 7.25)
    ax.set_xlabel(r"$R_{RUP}$ (km)", fontsize=16)
    ax.set_xticks([1, 10, 100], ["1", "10", "100"])
    ax.tick_params(labelsize=12)

fig.tight_layout()
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.91, 0.11, 0.03, 0.8])
cbar1 = fig.colorbar(cax1, cax=cbar_ax, label=r"$V_{S30}$ (m/s)", pad=0.02, )

# Add text boxes
axs[0].text(60, 6.9, "Irpinia\n(1980)", fontsize=14, va="center", bbox={"facecolor":"w", "alpha": 0.8})
axs[0].text(50, 6.56, "Central Italy\n(Oct. 2016)", fontsize=14, va="center", bbox={"facecolor":"w", "alpha": 0.8})
axs[0].text(60, 6.17, "L'Aquila\n(2009)", fontsize=14, va="center", bbox={"facecolor":"w", "alpha": 0.8})
axs[0].text(1.1, 6.0, "Emilia-\nRomagna\n(2012)", fontsize=14, va="center", bbox={"facecolor":"w", "alpha": 0.8})

# plt.savefig(...)

### Compare the variation in $\delta B_e$ with period (IMT)

In [ ]:
# The event IDs are different between the two data sets, indicate the common events and
# assign a pretty name
event_id_mapping = [
    # ESM (obs) ID, BBSpeedSet ID, "pretty" name
    ("EMSC-20161030_0000029", "ITA_2016.10.30",  "Central Italy (Oct. 2016)"), 
    ("IT-2012-0011", "ITA_2012.05.29", "Emilia-Romagna (2012)"),     
    ("IT-2009-0009", "ITA_2009.04.06", "L'Aquila (2009)"),
    ("IT-1980-0012", "ITA_1980.11.23_Mw6.9", "Irpinia (1980)"), 
]

# Extract just the inter-event residual columns
imt_columns = {}
for col in residuals_observations:
    if "inter_event_residual" not in col:
        continue
    imt, resid_type, gmm = col.split()
    if col.startswith("PGA"):
        imt_columns[col] = (imt, 0.01)  # Assigning PGA a period (just for plotting)
    elif col.startswith("PGV"):
        imt_columns[col] = (imt, 0.005)  # Assiging PGV a period (also just for plotting)
    elif col.startswith("SA("):
        imt_val = col.split()[0]
        imt_columns[col] = (imt, float(imt_val.replace("SA(", "").replace(")", "")))

# Built the dataframes containing the spectral periods (IMTs), the GMM, the data type
# and the event IDs
full_dataframe = []
for evid_obs, evid_sims, ev_name in event_id_mapping:
    idx_obs = residuals_observations["input rupture_parameter evt_id"] == evid_obs
    idx_sim = residuals_simulations["input rupture_parameter evt_id"] == evid_sims
    for gmm, gmm_label in gmms.items():
        for col, (imt, period) in imt_columns.items():
            # Get the values for the corresponding GMM and period
            yvals = residuals_observations[idx_obs][f"{imt} inter_event_residual {gmm}"]
            n = yvals.shape[0]
            df_obs = pd.DataFrame({
                "T": period * np.ones(n),
                "dBe": yvals,
                "GMM": [gmm_label] * n,
                "Type": ["Observation"] * n,
                "Event": [ev_name] * n
            })
            full_dataframe.append(df_obs)
            # Now repeat for simulations
            yvals = residuals_simulations[idx_sim][f"{imt} inter_event_residual {gmm}"]
            n = yvals.shape[0]
            df_sim = pd.DataFrame({
                "T": period * np.ones(n),
                "dBe": yvals,
                "GMM": [gmm_label] * n,
                "Type": ["Simulation"] * n,
                "Event": [ev_name] * n
            })
            full_dataframe.append(df_sim)
full_dataframe = pd.concat(full_dataframe, axis=0, ignore_index=True)
full_dataframe.drop_duplicates(inplace=True, ignore_index=True)
full_dataframe

To compare the $\delta B_e$ distributions between observation and simulation for each of the four events we limit our analysis to just one GMM. Here we use the ESHM20 GMM as it showed a minimal amount of bias in the observation datasets (alternative GMMs could also be considered).

In [ ]:
# Now plot dBE with period for each of the four events
selected_gmm = "ESHM20"
with sns.plotting_context("paper", font_scale=2.0):
    with sns.axes_style("darkgrid", ):
        fgrid = sns.lmplot(full_dataframe[full_dataframe["GMM"] == selected_gmm],
                           x="T",
                           y="dBe",
                           hue="Type",
                           col="Event",
                           row="GMM",
                           markers=["o", "s"],
                           fit_reg=False,
                           palette=["tab:blue", "tab:orange"],
                           scatter_kws={"s": 80},
                           facet_kws={"sharex": True, "sharey":True},
                          )
        fgrid.set(
            xlim=(0.004, 5.0),
            ylim=(-3.5, 3.5),      
            xscale="log"
        )
        fgrid.set_axis_labels("Period (s)", r"$\delta B_{e}$")
        for ax in fgrid.axes.flatten():
            ax.title.set_text(ax.title.get_text().replace("|", "\n"))
            ax.set_xticks(
                [0.005,  0.01,  0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0],
                ["PGV", "PGA", "0.05", "0.1", "0.2", "0.5", "1", "2", "5"],
                rotation=-40
            )
        fgrid.tight_layout(pad=0.15)

# To create the plot for Figure 8
# plt.savefig(
#     "./Figure_8_dBe_period_selected_events.jpg",
#     format="jpg",
#     dpi=300,
#     bbox_inches="tight"
# )

### Compare the distributions of $\delta W$ for the four events

In [ ]:
full_dataframe = []
for evid_obs, evid_sims, ev_name in event_id_mapping:
    idx_obs = residuals_observations["input rupture_parameter evt_id"] == evid_obs
    idx_sim = residuals_simulations["input rupture_parameter evt_id"] == evid_sims
    for gmm, gmm_label in gmms.items():
        for imt in ["PGV", "PGA", "SA(0.2)", "SA(2.0)"]:
            # Get the values for the corresponding GMM and period
            xvals = residuals_observations[idx_obs]["input distance_measure rrup"]           
            yvals = residuals_observations[idx_obs][f"{imt} intra_event_residual {gmm}"]
            n = yvals.shape[0]
            df_obs = pd.DataFrame({
                "rrup": xvals,
                "dW": yvals,
                "IMT": [imt] * n,
                "GMM": [gmm_label] * n,
                "Type": ["Observation"] * n,
                "Event": [ev_name] * n
            })
            full_dataframe.append(df_obs)
            # Now repeat for simulations
            xvals = residuals_simulations[idx_sim]["input distance_measure rrup"]  
            yvals = residuals_simulations[idx_sim][f"{imt} intra_event_residual {gmm}"]
            n = yvals.shape[0]
            df_sim = pd.DataFrame({
                "rrup": xvals,
                "dW": yvals,
                "IMT": [imt] * n,
                "GMM": [gmm_label] * n,
                "Type": ["Simulation"] * n,
                "Event": [ev_name] * n
            })
            full_dataframe.append(df_sim)
full_dataframe = pd.concat(full_dataframe, axis=0, ignore_index=True)
full_dataframe.drop_duplicates(inplace=True, ignore_index=True)
full_dataframe

In [ ]:
with sns.plotting_context("paper", font_scale=2.0):
    with sns.axes_style("darkgrid", ):
        fgrid = sns.lmplot(full_dataframe[full_dataframe["GMM"] == "ESHM20"],
                           x="rrup",
                           y="dW",
                           hue="Type",
                           col="IMT",
                           row="Event",
                           markers=["o", "s"],
                           fit_reg=True,
                           palette=["tab:blue", "tab:orange"],
                           scatter_kws={"s": 70},
                           facet_kws={"sharex": False, "sharey":True},
                          )
        fgrid.set(
            xlim=(0, 45.0),
            ylim=(-3.3, 3.3),
            xscale="linear"
        )
        fgrid.set_axis_labels(r"$R_{RUP}$ (km)", r"$\delta W_{es}$")
        for ax in fgrid.axes.flatten():
            ax.title.set_text(ax.title.get_text().replace("|", "\n"))
        fgrid.tight_layout()

# To create the plot for Figure 9
# plt.savefig(
#     "./Figure_9_dW_comparison_selected_events.jpg",
#     format="jpg",
#     dpi=300,
#     bbox_inches="tight"
# )